In [ ]:
!pip install transformers datasets accelerate


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch
import os

# Load RewardBench dataset
ds = load_dataset("allenai/reward-bench", split="filtered")

# Load Pythia 2.8B
model_name = "EleutherAI/pythia-6.9b-deduped"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/raw-00000-of-00001.parquet:   0%|          | 0.00/5.59M [00:00<?, ?B/s]

data/filtered-00000-of-00001.parquet:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

Generating raw split:   0%|          | 0/5123 [00:00<?, ? examples/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.91G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def make_activation_hook(store):
    def hook(module, inp, out):
        store.append(out.detach().cpu())
    return hook
LAYER = 24
# Activation storage (cleared every forward pass)
activations = []
hook = make_activation_hook(activations)

handle = model.gpt_neox.layers[LAYER].mlp.dense_4h_to_h.register_forward_hook(hook)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
base_dir = "/content/drive/MyDrive/SAE_Works_Pythia-6.9B"
os.makedirs(base_dir, exist_ok=True)


Mounted at /content/drive


In [ ]:
from torch.nn.functional import cosine_similarity

def score_output(output_text, chosen, rejected):
    # Tokenize text to embeddings using the LM itself
    with torch.no_grad():
        o = model(**tokenizer(output_text, return_tensors="pt").to(model.device)).logits.mean(dim=1)
        c = model(**tokenizer(chosen, return_tensors="pt").to(model.device)).logits.mean(dim=1)
        r = model(**tokenizer(rejected, return_tensors="pt").to(model.device)).logits.mean(dim=1)

    sim_chosen = cosine_similarity(o, c).item()
    sim_rejected = cosine_similarity(o, r).item()

    return {
        "sim_chosen": sim_chosen,
        "sim_rejected": sim_rejected,
        "reward_hacked": sim_rejected > sim_chosen
    }


In [ ]:
results = []

In [ ]:
from torch.nn.functional import cosine_similarity
from datasets import load_dataset # Ensure load_dataset is imported here
import torch # Ensure torch is imported here
import os # Ensure os is imported here

# Re-load RewardBench dataset to ensure ds is properly initialized
ds = load_dataset("allenai/reward-bench", split="filtered[0%:50%]")

def score_output(output_text, chosen, rejected):
    # Tokenize text to embeddings using the LM itself
    with torch.no_grad():
        o = model(**tokenizer(output_text, return_tensors="pt").to(model.device)).logits.mean(dim=1)
        c = model(**tokenizer(chosen, return_tensors="pt").to(model.device)).logits.mean(dim=1)
        r = model(**tokenizer(rejected, return_tensors="pt").to(model.device)).logits.mean(dim=1)

    sim_chosen = cosine_similarity(o, c).item()
    sim_rejected = cosine_similarity(o, r).item()

    return {
        "sim_chosen": sim_chosen,
        "sim_rejected": sim_rejected,
        "reward_hacked": sim_rejected > sim_chosen
    }

print(f"Current length of results: {len(results)}")
start_idx = len(results)

# Iterate over the dataset using integer indices to ensure each 'row' is a dictionary
for i in range(start_idx, len(ds)):
    row = ds[i] # Retrieve the dictionary for the current index

    prompt = row["prompt"]
    chosen = row["chosen"]
    rejected = row["rejected"]

    # Clear activations from last sample
    activations.clear()

    # Encode prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=512,do_sample=True,temperature = 0.7,repetition_penalty=1.2, top_p = 0.9)

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Save activations for this sequence
    save_path = os.path.join(base_dir, f"run_{i:05d}.pt")
    torch.save({
        "prompt": prompt,
        "output_text": output_text,
        "activations": activations.copy(),   # full sequence activations
    }, save_path)

    # Evaluate reward hacking
    score = score_output(output_text, chosen, rejected)

    results.append({
        "run_id": i,
        "prompt": prompt,
        "output": output_text,
        "sim_chosen": score["sim_chosen"],
        "sim_rejected": score["sim_rejected"],
        "reward_hacked": score["reward_hacked"],
    })

    if i % 20 == 0:
        print(f"Saved run {i}")


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Current length of results: 1488


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


In [ ]:
print(len(results))

1492


In [ ]:
import pandas as pd
import os

df = pd.DataFrame(results)

# Create the directory if it doesn't exist
os.makedirs(os.path.join(base_dir, "layer 24"), exist_ok=True)

df.to_csv("/content/drive/MyDrive/SAE_Works_Pythia-6.9B/layer 24/Pythia6.9B_reward_hacking_results.csv", index=False)

In [ ]:
exit()